references
- https://sagemaker-examples.readthedocs.io/en/latest/introduction_to_applying_machine_learning/xgboost_customer_churn/xgboost_customer_churn_outputs.html

In [1]:
import sys

!{sys.executable} -m pip install sagemaker pandas numpy matplotlib boto3 scikit-learn

  Using cached matplotlib-3.10.9-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (52 kB)
  Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached contourpy-1.3.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.63.0-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (118 kB)
  Using cached kiwisolver-1.5.0-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl.metadata (5.1 kB)
  Using cached pillow-12.2.0-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl

In [3]:
import sagemaker

sess = sagemaker.Session()
bucket = sess.default_bucket()
prefix = "sagemaker/xgboost-iris"

# Define IAM role
import boto3
import re
from sagemaker import get_execution_role

role = get_execution_role()

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import io
import os
import sys
import time
import json
import boto3
from IPython.display import display
from time import strftime, gmtime
from sagemaker.inputs import TrainingInput
from sagemaker.serializers import CSVSerializer

In [5]:
s3 = boto3.client("s3")
s3.download_file(f"sagemaker-us-east-1-200148130345", "Iris.csv", "Iris.csv")

In [54]:
df = pd.read_csv("./Iris.csv")
print(df.shape)
print(df.isnull().sum())
df.head()

(150, 6)
Id               0
SepalLengthCm    0
SepalWidthCm     0
PetalLengthCm    0
PetalWidthCm     0
Species          0
dtype: int64


,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


- https://docs.aws.amazon.com/sagemaker/latest/dg/xgboost-how-to-use.html#InputOutput-XGBoost

CSV의 경우, XGBoost 알고리즘은 무조건 first column을 label로 가정하고, header row를 필요로 하지 않는다.

In [55]:
le = LabelEncoder()
df['Species'] = le.fit_transform(df['Species'])

df = df.drop('Id', axis=1)
df = df.iloc[:, [4, 0, 1, 2, 3]]

In [56]:
df.head()

,Species,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm
0,0,5.1,3.5,1.4,0.2
1,0,4.9,3.0,1.4,0.2
2,0,4.7,3.2,1.3,0.2
3,0,4.6,3.1,1.5,0.2
4,0,5.0,3.6,1.4,0.2


In [63]:
train, test = train_test_split(df, test_size=0.2, random_state=42)

train.to_csv('train.csv', index=False, header=False)
test.to_csv('test.csv', index=False, header=False)

In [58]:
print(len(train), len(test))

120 30


In [59]:
boto3.Session().resource("s3").Bucket(bucket).Object(os.path.join(prefix, "train/train.csv")).upload_file("train.csv")
boto3.Session().resource("s3").Bucket(bucket).Object(os.path.join(prefix, "test/test.csv")).upload_file("test.csv")

INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole
INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole


In [60]:
s3_input_train = TrainingInput(
    s3_data="s3://{}/{}/train".format(bucket, prefix), content_type="csv"
)
s3_input_validation = TrainingInput(
    s3_data="s3://{}/{}/test/".format(bucket, prefix), content_type="csv"
)

In [61]:
container = sagemaker.image_uris.retrieve("xgboost", sess.boto_region_name, "1.7-1")
display(container)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


'683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1'

In [66]:
xgb = sagemaker.estimator.Estimator(
    container,
    role,
    instance_count=1,
    instance_type="ml.m4.xlarge",
    output_path="s3://{}/{}/output".format(bucket, prefix),
    sagemaker_session=sess,
)
xgb.set_hyperparameters(
    max_depth=5,
    eta=0.2,
    gamma=4,
    min_child_weight=6,
    subsample=0.8,
    verbosity=0,
    objective="multi:softmax",
    num_class=3,
    num_round=10,
    eval_metric="merror,mlogloss"
)

xgb.fit({"train": s3_input_train, "validation": s3_input_validation})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2026-06-16-07-56-07-854


2026-06-16 07:56:11 Starting - Starting the training job...
2026-06-16 07:56:25 Starting - Preparing the instances for training...
2026-06-16 07:56:49 Downloading - Downloading input data...
2026-06-16 07:57:34 Downloading - Downloading the training image......
2026-06-16 07:58:30 Training - Training image download completed. Training in progress.../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-06-16 07:58:45.591 ip-10-2-202-254.ec2.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-06-16 07:58:45.675 ip-10-2-202-254.ec2.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-06-16:07:58:45:INFO] Imported framework sagemaker_xgboost_container.trai

- accuracy: 1 - validation-merror

In [68]:
xgb_predictor = xgb.deploy(
    initial_instance_count=1, instance_type="ml.g4dn.xlarge", serializer=CSVSerializer()
)

INFO:sagemaker:Creating model with name: sagemaker-xgboost-2026-06-16-08-02-53-690
INFO:sagemaker:Creating endpoint-config with name sagemaker-xgboost-2026-06-16-08-02-53-690
INFO:sagemaker:Creating endpoint with name sagemaker-xgboost-2026-06-16-08-02-53-690


-----!

In [86]:
test_features = np.array([[28, 75000, 720, 3]])

# inference
result = xgb_predictor.predict(test_features)
print(result)  # 2.0 ==> Species: 2

b'2.0\n'
